# Download and refresh local price CSV files

Uses `download_adj_close_prices` from `data_loader.py` with provider fallback chain: `yfinance` -> `defeatbeta_api`.

It downloads each ticker in a specified date range and saves one CSV per ticker in `data/` with the same format used by the project (`Date,Adj Close`).

In [15]:
from __future__ import annotations

from pathlib import Path
import importlib
import time
import pandas as pd

import data_loader as _data_loader
importlib.reload(_data_loader)
download_adj_close_prices = _data_loader.download_adj_close_prices

In [16]:
TICKERS = [
    "CSSPX.MI",
    "AVUV",
    "IEF",
    "VUTY.AS",
    "SWDA.MI",
    "EIMI.MI",
    "CSNDX.MI",
    "IUSQ.MI",
    "VWCE.MI",
    "EXS1.MI",
    "MEUD.MI",
    "SMEA.MI",
    "XD9U.MI",
    "XMME.MI",
    "CSSX5E.MI",
    "EMUL.MI",
    "XLRE",
    "GOVT",
    "TLH",
    "LTPZ",
    "XTLT.TO",
    "UTHY",
    "TIP",
    "GLD",
    "IGLA.L",
    "XG7S.DE",
    "SEGA.MI",
    "VAGF.DE",
    "BTC=F",
    "SI=F",
    "CL=F",
    "CSH.PA",
    "PDBC",
    "VGLT",
    "VGSH",
    "VWRA.L",
]

In [17]:
# Configure your download window and tickers here
# Full ticker universe from the benchmark table (duplicates removed)

START_DATE = "2010-04-03"
END_DATE = "2020-04-14"

# Small delay between requests helps reduce provider throttling
COOLDOWN_SECONDS = 4

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Window: {START_DATE} -> {END_DATE}")
print(f"Output dir: {DATA_DIR.resolve()}")

Tickers: ['CSSPX.MI', 'AVUV', 'IEF', 'VUTY.AS', 'SWDA.MI', 'EIMI.MI', 'CSNDX.MI', 'IUSQ.MI', 'VWCE.MI', 'EXS1.MI', 'MEUD.MI', 'SMEA.MI', 'XD9U.MI', 'XMME.MI', 'CSSX5E.MI', 'EMUL.MI', 'XLRE', 'GOVT', 'TLH', 'LTPZ', 'XTLT.TO', 'UTHY', 'TIP', 'GLD', 'IGLA.L', 'XG7S.DE', 'SEGA.MI', 'VAGF.DE', 'BTC=F', 'SI=F', 'CL=F', 'CSH.PA', 'PDBC', 'VGLT', 'VGSH', 'VWRA.L']
Window: 2010-04-03 -> 2020-04-14
Output dir: /Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data


In [18]:
results = []

for i, ticker in enumerate(TICKERS, start=1):
    print(f"[{i}/{len(TICKERS)}] Downloading {ticker}...")
    status = "ok"
    rows = 0
    error = ""

    out_path = DATA_DIR / f"{ticker}.csv"

    try:
        req_start = pd.Timestamp(START_DATE)
        req_end = pd.Timestamp(END_DATE)
        one_day = pd.Timedelta(days=1)

        existing_series = None
        local_min = None
        local_max = None

        if out_path.exists():
            existing_df = pd.read_csv(out_path, parse_dates=["Date"])
            if not existing_df.empty and {"Date", "Adj Close"}.issubset(existing_df.columns):
                existing_series = (
                    existing_df.set_index("Date")["Adj Close"]
                    .astype(float)
                    .dropna()
                    .sort_index()
                )
                if not existing_series.empty:
                    local_min = existing_series.index.min()
                    local_max = existing_series.index.max()

        needs_left = local_min is None or req_start < local_min
        needs_right = local_max is None or req_end > local_max

        if not needs_left and not needs_right:
            # Requested window is fully covered locally: keep file untouched.
            final_series = existing_series
            print("    local data already covers requested window; no download needed")
        else:
            new_parts = []

            if needs_left:
                left_end = (local_min - one_day) if local_min is not None else req_end
                if req_start <= left_end:
                    left_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=req_start.strftime("%Y-%m-%d"),
                        end=left_end.strftime("%Y-%m-%d"),
                    )
                    left_series = left_prices[ticker].dropna()
                    new_parts.append(left_series)

            if needs_right:
                right_start = (local_max + one_day) if local_max is not None else req_start
                if right_start <= req_end:
                    right_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=right_start.strftime("%Y-%m-%d"),
                        end=req_end.strftime("%Y-%m-%d"),
                    )
                    right_series = right_prices[ticker].dropna()
                    new_parts.append(right_series)

            pieces = []
            if existing_series is not None and not existing_series.empty:
                pieces.append(existing_series)
            pieces.extend(new_parts)

            if not pieces:
                raise RuntimeError("No local data and no downloaded data available.")

            final_series = pd.concat(pieces).sort_index()
            final_series = final_series[~final_series.index.duplicated(keep="last")]

        # Force project-consistent on-disk shape: Date index + Adj Close column.
        out_df = final_series.to_frame(name="Adj Close")
        out_df.index.name = "Date"
        out_df.to_csv(out_path, date_format="%Y-%m-%d")

        rows = int(out_df.shape[0])
        print(f"    saved {rows} rows to {out_path}")
    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"    FAILED: {ticker}: {error}")

    results.append({"ticker": ticker, "status": status, "rows": rows, "error": error})

    if i < len(TICKERS):
        time.sleep(COOLDOWN_SECONDS)

summary = pd.DataFrame(results)
summary

[1/36] Downloading CSSPX.MI...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:324: RuntimeWarning: Local cache miss for symbols: ['CSSPX.MI']. Attempting online providers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()

1 Failed download:
['CSSPX.MI']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-04-03 00:00:00 -> 2010-05-19 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1270245600, endDate = 1274220000")')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:377: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['CSSPX.MI']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:386: RuntimeWarning: Yahoo Finance 

    FAILED: CSSPX.MI: No price data returned by yfinance for requested inputs.
[2/36] Downloading AVUV...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:324: RuntimeWarning: Local cache miss for symbols: ['AVUV']. Attempting online providers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()

1 Failed download:
['AVUV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-04-03 00:00:00 -> 2019-09-26 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1270267200, endDate = 1569470400")')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:377: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['AVUV']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:386: RuntimeWarning: Yahoo Finance batch respon

    FAILED: AVUV: No price data returned by yfinance for requested inputs.
[3/36] Downloading IEF...


...

    saved 4032 rows to data/VGSH.csv
[36/36] Downloading VWRA.L...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:324: RuntimeWarning: Local cache miss for symbols: ['VWRA.L']. Attempting online providers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()

1 Failed download:
['VWRA.L']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-04-03 00:00:00 -> 2019-07-23 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1270249200, endDate = 1563836400")')


    FAILED: VWRA.L: No price data returned by yfinance for requested inputs.


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:377: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['VWRA.L']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:386: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:179: RuntimeWarning: defeatbeta_api is not installed/importable, so fallback data cannot be fetched.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:407: RuntimeWarning: defeatbeta_api returned no usable rows for VWRA.L.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:412: RuntimeWarning: No fallback data source could recover VWRA.L after Yahoo batch failure (tried defeatb

,ticker,status,rows,error
0,CSSPX.MI,failed,0,No price data returned by yfinance for request...
1,AVUV,failed,0,No price data returned by yfinance for request...
2,IEF,ok,4032,
3,VUTY.AS,ok,2592,
4,SWDA.MI,ok,4200,
5,EIMI.MI,failed,0,No price data returned by yfinance for request...
6,CSNDX.MI,ok,4111,
7,IUSQ.MI,failed,0,No price data returned by yfinance for request...
8,VWCE.MI,failed,0,No price data returned by yfinance for request...
9,EXS1.MI,ok,4594,
